In [ ]:
!pip install -q pandas transformers sentencepiece accelerate

In [ ]:
import pandas as pd
import json
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [ ]:
test_cases = [
    {"id": 1, "feedback": "The product is amazing. I absolutely love it!", "expected": "Positive"},
    {"id": 2, "feedback": "Terrible experience. I will never buy again.", "expected": "Negative"},
    {"id": 3, "feedback": "The product arrived yesterday.", "expected": "Neutral"},
    {"id": 4, "feedback": "Excellent quality and fast delivery.", "expected": "Positive"},
    {"id": 5, "feedback": "The service was very disappointing.", "expected": "Negative"},
    {"id": 6, "feedback": "The package contains three items.", "expected": "Neutral"},
    {"id": 7, "feedback": "Very happy with my purchase!", "expected": "Positive"},
    {"id": 8, "feedback": "The app crashes every time I open it.", "expected": "Negative"},
    {"id": 9, "feedback": "The order was delivered on Monday.", "expected": "Neutral"},
    {"id": 10, "feedback": "Customer support was extremely helpful.", "expected": "Positive"},
    {"id": 11, "feedback": "Poor packaging and damaged product.", "expected": "Negative"},
    {"id": 12, "feedback": "The product is available in three colors.", "expected": "Neutral"},
    {"id": 13, "feedback": "Fantastic experience from start to finish.", "expected": "Positive"},
    {"id": 14, "feedback": "The product stopped working after one day.", "expected": "Negative"},
    {"id": 15, "feedback": "I received the replacement product today.", "expected": "Neutral"},
    {"id": 16, "feedback": "Great value for money. Highly recommended!", "expected": "Positive"},
    {"id": 17, "feedback": "Completely useless. Waste of money.", "expected": "Negative"},
    {"id": 18, "feedback": "The subscription renews every month.", "expected": "Neutral"},
    {"id": 19, "feedback": "I'm satisfied with the quality and performance.", "expected": "Positive"},
    {"id": 20, "feedback": "The delivery was late and nobody responded to my complaint.", "expected": "Negative"}
]

df = pd.DataFrame(test_cases)

print("Total test cases:", len(df))
display(df)

Total test cases: 20


,id,feedback,expected
0,1,The product is amazing. I absolutely love it!,Positive
1,2,Terrible experience. I will never buy again.,Negative
2,3,The product arrived yesterday.,Neutral
3,4,Excellent quality and fast delivery.,Positive
4,5,The service was very disappointing.,Negative
5,6,The package contains three items.,Neutral
6,7,Very happy with my purchase!,Positive
7,8,The app crashes every time I open it.,Negative
8,9,The order was delivered on Monday.,Neutral
9,10,Customer support was extremely helpful.,Positive


In [26]:
prompt_a = """
Classify the following customer feedback into one of these categories:
Positive, Negative.

Return only the category name.

Feedback: {feedback}
"""

In [ ]:
prompt_b = """
You are a customer feedback classification assistant.

Classify customer feedback into exactly one category:
Positive, Negative, or Neutral.

Guidelines:
- Positive: Expresses satisfaction, happiness, or appreciation.
- Negative: Expresses dissatisfaction, complaints, or frustration.
- Neutral: Contains factual information without clear positive or negative sentiment.

Examples:

Feedback: "I love this product. Excellent quality!"
Category: Positive

Feedback: "Very poor service. I am disappointed."
Category: Negative

Feedback: "The package arrived this morning."
Category: Neutral

Important:
- Consider the overall meaning of the feedback.
- Return only one category.
- Do not include explanations or additional text.

Feedback: {feedback}
"""

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1

llm = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Hugging Face model loaded successfully!")

Model loaded on: cpu


KeyError: 'The task text2text-generation has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Model loaded on:", device)

Model loaded on: cpu


In [ ]:
test_prompt = """
Classify the following customer feedback into:
Positive, Negative, or Neutral.

Return only the category name.

Feedback: I absolutely love this product!
"""

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=10,
    do_sample=False
)

result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Model output:", result)

Model output: Positive


In [ ]:
def get_prediction(prompt_template, feedback):

    prompt = prompt_template.format(feedback=feedback)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result.strip()

In [ ]:
prediction = get_prediction(
    prompt_a,
    "I absolutely love this product!"
)

print("Prediction:", prediction)

Prediction: Positive


In [ ]:
prediction = get_prediction(
    prompt_b,
    "The service was very disappointing."
)

print("Prediction:", prediction)

Prediction: Negative


In [27]:
results = []

for case in test_cases:

    feedback = case["feedback"]

    prediction_a = get_prediction(prompt_a, feedback)
    prediction_b = get_prediction(prompt_b, feedback)

    results.append({
        "id": case["id"],
        "feedback": feedback,
        "expected": case["expected"],
        "prompt_a": prediction_a,
        "prompt_b": prediction_b
    })

results_df = pd.DataFrame(results)

display(results_df)

,id,feedback,expected,prompt_a,prompt_b
0,1,The product is amazing. I absolutely love it!,Positive,Positive,Positive
1,2,Terrible experience. I will never buy again.,Negative,Negative,Negative
2,3,The product arrived yesterday.,Neutral,Positive,Positive
3,4,Excellent quality and fast delivery.,Positive,Positive,Positive
4,5,The service was very disappointing.,Negative,Negative,Negative
5,6,The package contains three items.,Neutral,Positive,Neutral
6,7,Very happy with my purchase!,Positive,Positive,Positive
7,8,The app crashes every time I open it.,Negative,Negative,Negative
8,9,The order was delivered on Monday.,Neutral,Positive,Positive
9,10,Customer support was extremely helpful.,Positive,Positive,Positive


In [28]:
print("Total test cases:", len(results_df))
print("Prompt A predictions:", results_df["prompt_a"].count())
print("Prompt B predictions:", results_df["prompt_b"].count())

Total test cases: 20
Prompt A predictions: 20
Prompt B predictions: 20


In [29]:
valid_labels = ["Positive", "Negative", "Neutral"]

def normalize_prediction(prediction):

    prediction = prediction.strip().capitalize()

    if prediction in valid_labels:
        return prediction

    return "Invalid"

results_df["prompt_a"] = results_df["prompt_a"].apply(normalize_prediction)
results_df["prompt_b"] = results_df["prompt_b"].apply(normalize_prediction)

display(results_df)

,id,feedback,expected,prompt_a,prompt_b
0,1,The product is amazing. I absolutely love it!,Positive,Positive,Positive
1,2,Terrible experience. I will never buy again.,Negative,Negative,Negative
2,3,The product arrived yesterday.,Neutral,Positive,Positive
3,4,Excellent quality and fast delivery.,Positive,Positive,Positive
4,5,The service was very disappointing.,Negative,Negative,Negative
5,6,The package contains three items.,Neutral,Positive,Neutral
6,7,Very happy with my purchase!,Positive,Positive,Positive
7,8,The app crashes every time I open it.,Negative,Negative,Negative
8,9,The order was delivered on Monday.,Neutral,Positive,Positive
9,10,Customer support was extremely helpful.,Positive,Positive,Positive


In [30]:
accuracy_a = (
    results_df["expected"] == results_df["prompt_a"]
).mean() * 100

accuracy_b = (
    results_df["expected"] == results_df["prompt_b"]
).mean() * 100

print(f"Prompt A Accuracy: {accuracy_a:.2f}%")
print(f"Prompt B Accuracy: {accuracy_b:.2f}%")

Prompt A Accuracy: 70.00%
Prompt B Accuracy: 85.00%


In [31]:
# Identify incorrect predictions

failures_a = results_df[
    results_df["expected"] != results_df["prompt_a"]
].copy()

failures_b = results_df[
    results_df["expected"] != results_df["prompt_b"]
].copy()

# Add prompt names

failures_a["prompt_name"] = "Prompt A - Basic"
failures_b["prompt_name"] = "Prompt B - Few-shot"

# Combine failures

failure_log = pd.concat(
    [failures_a, failures_b],
    ignore_index=True
)

display(
    failure_log[
        ["id", "feedback", "expected",
         "prompt_a", "prompt_b", "prompt_name"]
    ]
)

print("Total failures:", len(failure_log))

,id,feedback,expected,prompt_a,prompt_b,prompt_name
0,3,The product arrived yesterday.,Neutral,Positive,Positive,Prompt A - Basic
1,6,The package contains three items.,Neutral,Positive,Neutral,Prompt A - Basic
2,9,The order was delivered on Monday.,Neutral,Positive,Positive,Prompt A - Basic
3,12,The product is available in three colors.,Neutral,Positive,Neutral,Prompt A - Basic
4,15,I received the replacement product today.,Neutral,Positive,Positive,Prompt A - Basic
5,18,The subscription renews every month.,Neutral,Positive,Neutral,Prompt A - Basic
6,3,The product arrived yesterday.,Neutral,Positive,Positive,Prompt B - Few-shot
7,9,The order was delivered on Monday.,Neutral,Positive,Positive,Prompt B - Few-shot
8,15,I received the replacement product today.,Neutral,Positive,Positive,Prompt B - Few-shot


Total failures: 9
